# Figure 4,5,6,7 - Perturbation-specific Figures (Python portion)
(A) Signature Percentile Shift per perturbation
(B) Signature Distribution
(E) Disease DEG Overlap

Note: Mixscale heatmaps (C) and Mixscale Volcano Plots (D) are in R under Mixscale folder

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys 
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc 
import muon as mu

sys.path.append('../utils')

import signature_heatmaps as signature_heatmaps
import factor_labels as factor_labels


## Load Data

In [ ]:
data_dir = "<path to processed data>"

cite_6tf_path = os.path.join(data_dir, "cite_6tf_cleaned_revisions.h5mu")
cite_imgl_path = os.path.join(data_dir, "cite_imgl_cleaned_revisions.h5mu")
merged_6tf_path = os.path.join(data_dir, "adata_revisions_merged_6tf.h5ad")

In [ ]:
mdata_dict = {}
mdata_dict['cite_6tf'] = mu.read_h5mu(cite_6tf_path)
mdata_dict['cite_imgl'] = mu.read_h5mu(cite_imgl_path)

adata_dict = {}
adata_dict['merged_6tf'] = sc.read_h5ad(merged_6tf_path)
adata_dict['cite_6tf'] = mdata_dict['cite_6tf'].mod['rna'].copy()
adata_dict['cite_imgl'] = mdata_dict['cite_imgl'].mod['rna'].copy()

In [ ]:
print(mdata_dict['cite_6tf'])
print(mdata_dict['cite_imgl'])
print(adata_dict['merged_6tf'])

In [ ]:
signature_cols_ordered = ['homeostatic_score_ucell',
 'interferon_score_ucell',
 'chemokine_score_ucell',
 'antigen_presenting_score_ucell',
 'dam_score_ucell',
 'lipid_dam_score_ucell']

## Masking for analysis
to exclude ntc_g5 + foxk1_g2 + mixscale_cutoff >= 0

In [ ]:
guides_to_exclude = ['FOXK1_g2', 'non-targeting_g5']

adata_6tf_clean = adata_dict['merged_6tf'][~adata_dict['merged_6tf'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
adata_imgl_clean = adata_dict['cite_imgl'][~adata_dict['cite_imgl'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
adata_6tf_clean.shape, adata_imgl_clean.shape

In [ ]:
print(mdata_dict['cite_6tf'].shape, mdata_dict['cite_imgl'].shape)
mdata_6tf_clean = mdata_dict['cite_6tf'][~mdata_dict['cite_6tf'].mod['rna'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
mdata_imgl_clean = mdata_dict['cite_imgl'][~mdata_dict['cite_imgl'].mod['rna'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
mdata_6tf_clean.shape, mdata_imgl_clean.shape

In [ ]:
mixscale_col = "mixscale_score"

In [ ]:
adata_6tf_masked = adata_6tf_clean[adata_6tf_clean.obs[mixscale_col] >= 0].copy()
adata_imgl_masked = adata_imgl_clean[adata_imgl_clean.obs[mixscale_col] >= 0].copy()
adata_6tf_masked.shape, adata_imgl_masked.shape

## (A) Signature Percentile Shifts per Perturbation

  - In iTF - STAT2, DNMT1, SMAD3, IRF9
  - In iMGL - PRDM1, ZNF532

#### Plotting Percentile Shifts

In [ ]:
percentile_df_6tf_sign = pd.read_csv("figures/iTF/percentile_df_figA_iTF_signatures.csv", index_col = 0)
percentile_df_imgl_sign = pd.read_csv("figures/iMG/percentile_df_figA_iMG_signatures.csv", index_col = 0)

In [ ]:
genes_to_include = ['DNMT1', 'IRF9', 'PRDM1','SMAD3', 'STAT2', 'ZNF532']

In [ ]:
sns.set_theme("notebook", "white")
percentile_dict = {"iTF": percentile_df_6tf_sign, "iMG": percentile_df_imgl_sign}
out_dir = "<path to output directory>"
for gene in genes_to_include:
    for name, percentile_df_to_use in percentile_dict.items():
        fig_dir = f"{out_dir}/{name}/final/"
        percentile_df_gene = percentile_df_to_use[percentile_df_to_use['perturbed_gene'].isin(['NTC', gene])]
        ordered_list_gene = list(percentile_df_gene[percentile_df_gene['perturbed_guide'] != "NTC"].sort_values("MedianPct").Factor.unique())
        
        signature_heatmaps.draw_pointplot(percentile_df_gene, [gene], (12,3), by_guide=False, difference=True,
                             diff_type_name="Factor", nrows=1, ncols=2, 
                             palette = factor_labels.palette_dict_imgl if name == "iMG" else factor_labels.palette_dict_6tf,
                             orderlist = ordered_list_gene, 
                             filename=f"{fig_dir}/{gene}_signature_percentile_shift_figA",
                             filetype=[".svg", ".jpeg", ".pdf"])

## (B) Signature Distributions

In [ ]:
use_itf = ['DNMT1', 'STAT2', 'IRF9', 'SMAD3']
use_img = ['PRDM1', 'ZNF532']

In [ ]:
genes_to_include = use_itf + use_img

In [ ]:
for gene in genes_to_include:
    if gene in use_itf:
        sub_df = adata_6tf_masked[adata_6tf_masked.obs.perturbed_gene.isin(["NTC", gene])]
        fig_dir = f"{out_dir}/iTF/final"
    else:
        sub_df = adata_imgl_masked[adata_imgl_masked.obs.perturbed_gene.isin(["NTC", gene])]
        fig_dir = f"{out_dir}/iMG/final"

    gene_mask = sub_df.obs['perturbed_gene'] == gene
    ntc_mask = sub_df.obs['perturbed_gene'] == "NTC"

    gene_cells = sub_df[gene_mask]
    ntc_cells = sub_df[ntc_mask]

    sns.set_theme("poster", "white")
    for signature in signature_cols_ordered:
        fig = plt.figure(figsize=(5,5), dpi=300)
        sns.kdeplot(sub_df.obs,
                    x=signature,
                    hue="perturbed_guide",
                    legend=False,
                    palette=factor_labels.palette_dict_6tf if gene in use_itf else factor_labels.palette_dict_imgl,
                    fill=False,
                    linewidth = 6,
                    cumulative=True,
                    common_norm=False
                    )
        fig.savefig(f"{fig_dir}/{gene}_{signature}_histogram_norm_cumulative.svg")

# (E) Disease DEG Overlap

In [ ]:
deg_dir = "<path to Mixscale degs>"

deg_dict = {}
sig_deg_dict = {}
pval_threshold = 0.05
logfc_threshold = 0
genes = ['STAT2', 'DNMT1', 'SMAD3', 'IRF9', 'PRDM1', 'ZNF532']
for model in ['merged_6tf', 'cite_imgl']:
    sig_deg_dict[model] = {}
    deg_dict[model] = {}
    for gene in genes:
        if model == "merged_6tf":
            path = deg_dir + f"/{model}_mixscale_degs_all_rna_{gene}.csv"
        else:
            path = deg_dir + f"/{model}_mixscale_degs_sig_rna_{gene}.csv"
        if os.path.exists(path):
            deg_dict[model][gene] = pd.read_csv(path, index_col = 0)
            deg_dict[model][gene]['gene_ID'] = deg_dict[model][gene].index
            
            sig_deg_dict[model][gene] = {}
            sig_deg_dict[model][gene]['df'] = deg_dict[model][gene][deg_dict[model][gene]['adj_p_weight'] <= pval_threshold].copy()
            sig_deg_dict[model][gene]['negative'] = sig_deg_dict[model][gene]['df'][sig_deg_dict[model][gene]['df']['log2FC'] <= logfc_threshold].copy()
            sig_deg_dict[model][gene]['positive'] = sig_deg_dict[model][gene]['df'][sig_deg_dict[model][gene]['df']['log2FC'] > logfc_threshold].copy()

In [ ]:
h_clus_df = pd.read_csv("../../data/literature/humica_lit_signatures.csv")
h_clus_df['group'] = h_clus_df['Population'] + "-" + h_clus_df['Study']
group_gene_dict = {}
for group in h_clus_df['group'].unique():
    group_genes = set(h_clus_df[h_clus_df['group'] == group]['Gene'])
    group_gene_dict[group] = group_genes

In [ ]:
h_dis_df = pd.read_csv("../../data/literature/humica_disease_degs.csv", header=1)
h_dis_df = h_dis_df[h_dis_df['padj'] <= pval_threshold]
h_dis_df['direction'] = np.where(h_dis_df['log2FoldChange'] > 0, 'positive', 'negative')
h_dis_df['joint_group'] = h_dis_df['Group'] + "-" + h_dis_df['direction']

for group in h_dis_df['joint_group'].unique():
    group_genes = set(h_dis_df[h_dis_df['joint_group'] == group]['Gene'])
    group_gene_dict[group] = group_genes

In [ ]:
h_dis_dam_df = pd.read_csv("../../data/literature/humica_disease_degs_dam_clusters.csv")

h_dis_dam_df = h_dis_dam_df[h_dis_dam_df['padj'] <= pval_threshold]
h_dis_dam_df['direction'] = np.where(h_dis_dam_df['log2FoldChange'] > 0, 'positive', 'negative')
h_dis_dam_df['joint_group'] = h_dis_dam_df['Group'] + "-" + h_dis_dam_df['direction'] + "-" + "dam_cluster"

for group in h_dis_dam_df['joint_group'].unique():
    group_genes = set(h_dis_dam_df[h_dis_dam_df['joint_group'] == group]['Gene'])
    group_gene_dict[group] = group_genes

In [ ]:
from scipy.stats import fisher_exact

overlap_results = []

for model in sig_deg_dict:
    for gene in sig_deg_dict[model]:
        for direction in ['negative', 'positive']:
            deg_genes = set(sig_deg_dict[model][gene][direction]['gene_ID']) if not sig_deg_dict[model][gene][direction].empty else set()
            total_genes = set()
            for g in deg_dict[model]:
                total_genes.update(deg_dict[model][g].index)
            for group, group_genes in group_gene_dict.items():
                overlap_genes = deg_genes & group_genes
                overlap_count = len(overlap_genes)
                group_size = len(group_genes)
                deg_size = len(deg_genes)
                total_size = len(total_genes)
                # Build contingency table
                a = overlap_count
                b = group_size - overlap_count
                c = deg_size - overlap_count
                d = total_size - (a + b + c)
                table = [[a, b], [c, d]]
                _, fisher_p = fisher_exact(table, alternative='greater')
                overlap_percentage = (overlap_count / group_size * 100) if group_size > 0 else 0
                overlap_results.append({
                    'model': model,
                    'perturbed_gene': gene,
                    'direction': direction,
                    'group': group,
                    'overlap_count': overlap_count,
                    'overlap_percentage': overlap_percentage,
                    'overlap_genes': list(overlap_genes),
                    "fisher_p": fisher_p
                })

overlap_df = pd.DataFrame(overlap_results)

In [ ]:
overlap_df.to_csv('<path to output dir with overlap dataframe>')

# - Finished Python Portion of Figure 3-6 (perturbation specific) -